In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.nn import (
    MessagePassing,
    GCNConv,
    SAGEConv,
    GATConv,
    GINConv
)
import time
import pandas as pd

# ----------------------------------------------------------------------
# 1. Base Message Passing (Vanilla GNN)
# ----------------------------------------------------------------------
class VanillaGNNConv(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')  # Sum aggregation
        self.lin = nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index):
        return self.propagate(edge_index, x=x)

    def message(self, x_j):
        return x_j

    def update(self, aggr_out):
        return self.lin(aggr_out)

# ----------------------------------------------------------------------
# 2. Unified Model Architecture Wrapper
# ----------------------------------------------------------------------
class BenchmarkGNN(nn.Module):
    def __init__(self, model_type, in_channels, hidden_channels, out_channels, heads=8, dropout=0.5):
        super().__init__()
        self.model_type = model_type
        self.dropout = dropout

        if model_type == 'Vanilla GNN':
            self.conv1 = VanillaGNNConv(in_channels, hidden_channels)
            self.conv2 = VanillaGNNConv(hidden_channels, out_channels)
        elif model_type == 'GCN':
            self.conv1 = GCNConv(in_channels, hidden_channels)
            self.conv2 = GCNConv(hidden_channels, out_channels)
        elif model_type == 'GraphSAGE':
            self.conv1 = SAGEConv(in_channels, hidden_channels, aggr='mean')
            self.conv2 = SAGEConv(hidden_channels, out_channels, aggr='mean')
        elif model_type == 'GAT':
            self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
            self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=dropout)
        elif model_type == 'GIN':
            mlp1 = nn.Sequential(
                nn.Linear(in_channels, hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, hidden_channels)
            )
            mlp2 = nn.Sequential(
                nn.Linear(hidden_channels, hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, out_channels)
            )
            self.conv1 = GINConv(mlp1, train_eps=True)
            self.conv2 = GINConv(mlp2, train_eps=True)
        else:
            raise ValueError(f"Unknown model_type: {model_type}")

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

# ----------------------------------------------------------------------
# 3. Training & Evaluation Routines
# ----------------------------------------------------------------------
def train_epoch(model, data, optimizer, criterion):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def evaluate(model, data):
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=-1)

    accs = []
    for mask in [data.train_mask, data.val_mask, data.test_mask]:
        correct = pred[mask].eq(data.y[mask]).sum().item()
        accs.append(correct / mask.sum().item())
    return accs  # [train_acc, val_acc, test_acc]

# ----------------------------------------------------------------------
# 4. Main Benchmark Execution
# ----------------------------------------------------------------------
def run_benchmark():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running benchmarks on: {device}\n")

    # Load Cora dataset
    dataset = Planetoid(root='/tmp/Cora', name='Cora', transform=T.NormalizeFeatures())
    data = dataset[0].to(device)

    models_to_test = ['Vanilla GNN', 'GCN', 'GraphSAGE', 'GAT', 'GIN']
    results = []

    hidden_dim = 64
    epochs = 200
    lr = 0.01
    weight_decay = 5e-4

    for name in models_to_test:
        torch.manual_seed(42)
        model = BenchmarkGNN(
            model_type=name,
            in_channels=dataset.num_features,
            hidden_channels=hidden_dim,
            out_channels=dataset.num_classes,
            heads=8,
            dropout=0.5
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        criterion = nn.NLLLoss()

        best_val_acc = 0.0
        best_test_acc = 0.0

        start_time = time.time()
        for epoch in range(1, epochs + 1):
            loss = train_epoch(model, data, optimizer, criterion)
            train_acc, val_acc, test_acc = evaluate(model, data)

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_test_acc = test_acc

        total_time = time.time() - start_time
        param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

        results.append({
            'Model': name,
            'Trainable Params': param_count,
            'Best Val Acc (%)': round(best_val_acc * 100, 2),
            'Test Acc @ Best Val (%)': round(best_test_acc * 100, 2),
            'Time (s)': round(total_time, 2)
        })

    df = pd.DataFrame(results)
    print("=" * 65)
    print("                 BENCHMARK RESULTS ON CORA")
    print("=" * 65)
    print(df.to_string(index=False))

if __name__ == '__main__':
    run_benchmark()